In [1]:
import time
import hashlib
from typing import Dict, Any
from collections import defaultdict

from langchain_community.chat_models import ChatOllama


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [16]:
llm_fast = ChatOllama(model="llama3", temperature=0)
llm_smart = ChatOllama(model="llama3", temperature=0)



In [17]:
class Request:
    def __init__(self, user_id: str, prompt: str):
        self.user_id = user_id
        self.prompt = prompt


class Response:
    def __init__(self, text: str, meta: Dict[str, Any]):
        self.text = text
        self.meta = meta


In [18]:
def validate_request(req: Request):
    if not req.prompt.strip():
        raise ValueError("Empty prompt")

    banned = ["illegal", "violence"]
    if any(b in req.prompt.lower() for b in banned):
        raise ValueError("Unsafe content")


In [19]:
def route_model(prompt: str):
    if len(prompt) < 200:
        return llm_fast, "fast"
    return llm_smart, "smart"


In [20]:
cache = {}

def cache_key(prompt: str) -> str:
    return hashlib.sha256(prompt.encode()).hexdigest()


In [21]:
def cached_inference(llm, prompt: str):
    key = cache_key(prompt)
    if key in cache:
        return cache[key], True

    result = llm.invoke(prompt).content
    cache[key] = result
    return result, False


In [22]:
metrics = defaultdict(list)

def record(metric: str, value: float):
    metrics[metric].append(value)


In [23]:
def safe_invoke(llm, prompt: str, timeout=5):
    start = time.time()
    try:
        result = llm.invoke(prompt).content
    except Exception:
        return "Model error", "error"

    latency = time.time() - start
    if latency > timeout:
        return "Timed out", "timeout"

    return result, "ok"


In [24]:
def handle_request(req: Request) -> Response:
    start = time.time()

    validate_request(req)

    llm, route = route_model(req.prompt)

    text, cached = cached_inference(llm, req.prompt)

    latency = time.time() - start

    record("latency", latency)
    record("cached", int(cached))

    return Response(
        text=text,
        meta={
            "route": route,
            "cached": cached,
            "latency": latency
        }
    )


In [25]:
req = Request(user_id="u1", prompt="Explain inference optimization briefly.")
resp = handle_request(req)

print(resp.text)
print(resp.meta)


Inference optimization is the process of improving the efficiency and speed of machine learning models, particularly those that rely on probabilistic reasoning or Bayesian networks, when making predictions or inferences.

The goal of inference optimization is to reduce the computational complexity and time required for model evaluation, while maintaining or even improving the accuracy of the predictions. This is achieved by:

1. **Pruning**: Eliminating unnecessary nodes or variables from the model, reducing the number of computations needed.
2. **Approximation**: Using simplified representations or approximations of complex calculations, such as Monte Carlo methods or variational inference.
3. **Parallelization**: Distributing the computation across multiple processors or cores to speed up processing.
4. **Caching**: Storing intermediate results or model parameters to avoid redundant computations.

Inference optimization is crucial in applications where models need to make predictions

In [26]:
handle_request(req).meta


{'route': 'fast', 'cached': True, 'latency': 0.0}

In [27]:
print("Avg latency:", sum(metrics["latency"]) / len(metrics["latency"]))
print("Cache hit rate:", sum(metrics["cached"]) / len(metrics["cached"]))


Avg latency: 35.153653621673584
Cache hit rate: 0.5
